## Algorithm (v2): Building a Feature-Importance Model

This notebook uses a gold-standard dataset to estimate the relative importance of bibliographic attributes (for example, DOI and title) when determining whether two records (for example, journal articles or books) are duplicates.

The workflow evaluates multiple machine-learning approaches to build a classifier with high sensitivity and specificity, with particular emphasis on minimising false positives.

The resulting weights are intended for integration into `WeightSettings` (`BaseModel`) and combined into a scoring model that outputs duplicate probabilities for record pairs in future datasets.

This version builds on earlier work (v1). Since v1, the project has:
- improved field-matching functions in the deduper;
- added early-stop rules to improve efficiency; and
- added safeguards to reduce false positives.

The goal of this notebook is to finalise the v2 algorithm for implementation.


### Import Libraries and Configure Paths

In [1]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from loguru import logger
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, "..")

from algorithm.algorithm_development import (
    build_blocked_pairs_from_df,
    build_record_cache,
    read_process_data_from_file,
    score_pairs_with_early_stop,
)

from app.candidate_selection import (
 BLOCK_RULES
)

REPO_DIR = Path.cwd().parent
RAW_DATA = REPO_DIR / "notebooks/data/srsr_data.csv"
RESULTS_DIR = REPO_DIR / "notebooks" / "results"
GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

logger.remove()
logger.add(sys.stderr, level="WARNING")
logger.disable("app")
logger.disable("app.dedupe")
logger.disable("app.data_models")
logger.disable("app.determine_weights")
logger.disable("app.early_stop")

### Prepare gold standard dataset
We load the ASySD gold-standard dataset from `srsr_data.csv`, standardise column names, and align key fields (for example, `author` → `authors`, `number` → `issue`).  

We then generate candidate record pairs using the configured `BLOCK_RULES`, applying field-aware normalisation (for example DOI, pages, and numeric metadata) before block matching.

Each blocked pair is labelled as duplicate (`is_dupe = 1`) when both records share the same `duplicateid`; otherwise it is labelled non-duplicate (`is_dupe = 0`). This produces a gold-standard candidate-pair dataset containing both true duplicate pairs and challenging “close” non-duplicate pairs for model training and evaluation.

In [2]:
df = read_process_data_from_file(RAW_DATA, encoding="latin-1")
print(df.shape)
df.head()

C:\Users\khair\AppData\Local\Temp\ipykernel_39924\211439177.py:1: DeprecationWarning: read_process_data_from_file is deprecated. Use app.import_references.load_reference_csv instead.
  df = read_process_data_from_file(RAW_DATA, encoding="latin-1")


(53001, 14)


,unnamed: 0,ï..,authors,year,journal,doi,title,pages,volume,issue,abstract,recordid,isbn,duplicateid
0,1,20363,Slavin S. A.Lin S. J.,2012.0,Plast Reconstr Surg,10.1097/PRS.0b013e31825f23ca,THE USE OF ACELLULAR DERMAL MATRICES IN REVISI...,70s-85s,130,5 Suppl 2,BACKGROUND: The use of acellular dermal matric...,15153,0032-1052,15153
1,2,23210,Wang K. K.Donahue T. R.Haber G. B.DeWitt J. M....,2017.0,Gastroenterology,NaN,THE USE OF PHOTODYNAMIC THERAPY IN PANCREATIC ...,S498,152 (5 Supplement 1),NaN,Background: A scientific panel of experts was ...,29964,1528-0012,29964
2,3,23459,Watson L. I.Armon M. P.,2004.0,Cochrane Database Syst Rev,10.1002/14651858.CD002783.pub2,THROMBOLYSIS FOR ACUTE DEEP VEIN THROMBOSIS,Cd002783,NaN,4,BACKGROUND: Standard treatment for deep vein t...,22076,1361-6137,22076
3,4,7212,Gagyor I.Madhok V. B.Daly F.Somasundara D.Sull...,2015.0,Cochrane Database Syst Rev,10.1002/14651858.CD001869.pub5,WITHDRAWN. ANTIVIRAL TREATMENT FOR BELL'S PALS...,Cd001869,NaN,5,BACKGROUND: Corticosteroids are widely used in...,10934,1361-6137,10934
4,5,46468,Wang E. E.Tang N. K.,2007.0,Cochrane database of systematic reviews (Online),NaN,WITHDRAWN: IMMUNOGLOBULIN FOR PREVENTING RESPI...,CD001725,NaN,3,BACKGROUND: Respiratory Syncytial virus the mo...,36237,1469-493X (electronic)\r\n1469-493X,20457


### Generate blocked candidate pairs and labels

Using the cleaned gold-standard dataframe (`df`), we construct candidate record pairs by applying each rule in `BLOCK_RULES`.

For each rule:
- relevant fields are normalised with `norm_value`;
- records sharing the same block key are grouped; and
- all within-group combinations are generated as candidate pairs.

Each pair is then labelled:
- `is_dupe = 1` if both records share the same `duplicateid`;
- `is_dupe = 0` otherwise.

We also track which blocking rule(s) produced each pair to support diagnostics and later error analysis.

In [3]:
all_pairs_df = build_blocked_pairs_from_df(
    df,
    block_rules=BLOCK_RULES,
    id_column="recordid",
    dup_column="duplicateid",
    include_block_rules=True,
 )

# Overall totals
n_dupes = all_pairs_df["is_dupe"].sum()
n_non_dupes = (all_pairs_df["is_dupe"] == 0).sum()
print(f"Total pairs: {len(all_pairs_df):,}  |  dupes: {n_dupes:,}  |  non-dupes: {n_non_dupes:,}")

Total pairs: 1,019,402  |  dupes: 22,694  |  non-dupes: 996,708


This section creates a reproducible training subset of blocked pairs before scoring:

- All duplicate pairs (`dupes_sample`) and a sample of non-duplicate pairs (`non_dupes_sample`) is drawn from `all_pairs_df` using `GLOBAL_SEED`.
- The non-duplicate sample is set to a **4:1 ratio** relative to duplicates to preserve challenging negatives while keeping runtime manageable.
- Both sets are combined and shuffled into `sample_pairs_df`.
- A lightweight `record_cache` is then built **only for record IDs present in the sample** (`id_a`/`id_b`), reducing memory and validation overhead during scoring.

The next cells then run feature scoring and persist outputs for reproducibility:

- `score_pairs(sample_pairs_df, record_cache)` computes per-field similarity features for each sampled pair, records early-stop rule hits, and returns `scored_df`.
- The notebook saves both intermediate datasets to `RESULTS_DIR`:
    - `sample_pairs_seed_{GLOBAL_SEED}.csv` (exact sampled pair set)
    - `scored_pairs_seed_{GLOBAL_SEED}.csv` (scored feature table)

This ensures the exact sampled inputs and derived features can be reloaded and audited in later runs.


In [4]:
from app.dedupe import WEIGHTS

SAMPLE_N_DUPES = 22694
SAMPLE_N_NON_DUPES = 22694*4

dupes_sample = all_pairs_df[all_pairs_df["is_dupe"] == 1].sample(
    n=min(SAMPLE_N_DUPES, (all_pairs_df["is_dupe"] == 1).sum()), random_state=42
)
non_dupes_sample = all_pairs_df[all_pairs_df["is_dupe"] == 0].sample(
    n=min(SAMPLE_N_NON_DUPES, (all_pairs_df["is_dupe"] == 0).sum()), random_state=42
)
sample_pairs_df = pd.concat([dupes_sample, non_dupes_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

# Build cache only for IDs in sample
sample_ids = set(sample_pairs_df["id_a"]).union(sample_pairs_df["id_b"])
record_cache, validation_errors = build_record_cache(df, id_column="recordid", include_ids=sample_ids)
if validation_errors:
    print(f"Skipped {validation_errors} invalid records while building the cache")

print(f"Sample: {len(dupes_sample):,} dupes | {len(non_dupes_sample):,} non-dupes | cache size: {len(record_cache):,}")

Sample: 22,694 dupes | 90,776 non-dupes | cache size: 44,759


In [5]:
scored_df, early_stop_summary_df = score_pairs_with_early_stop(
    sample_pairs_df,
    record_cache
 )

stopped_total = int(early_stop_summary_df["count"].sum()) if not early_stop_summary_df.empty else 0
print(f"Early stop triggered for {stopped_total:,} / {len(scored_df):,} scored pairs")

if not early_stop_summary_df.empty:
    display(early_stop_summary_df)
else:
    print("No early-stop rules were triggered.")

scored_df

100%|██████████| 113470/113470 [04:00<00:00, 472.70it/s]


Early stop triggered for 90,900 / 113,470 scored pairs


,early_stop_rule,count
0,doi_and_pages_mismatch,56562
1,partial_ratio_too_low,34129
2,title_with_metadata_mismatch,172
3,year_gap_with_abstract_conflict,24
4,doi_pub_version_mismatch,13


,id_a,id_b,is_dupe,early_stop_rule,doi,title,authors,year,journal,pages,abstract,volume,issue
0,28284,28291,0,doi_and_pages_mismatch,0.0,0.234483,0.529493,1.0,0.515873,0.0,0.238199,1.00000,0.0
1,5483,28925,0,doi_and_pages_mismatch,0.0,0.195652,0.489984,1.0,0.462500,0.0,0.271598,1.00000,0.0
2,20975,55985,1,None,0.0,1.000000,0.879524,1.0,0.850000,1.0,0.000000,1.00000,1.0
3,10510,31824,1,None,1.0,1.000000,1.000000,1.0,1.000000,1.0,0.970213,1.00000,1.0
4,34437,53764,0,partial_ratio_too_low,0.0,0.240310,0.452753,1.0,0.279956,0.0,0.000000,1.00000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
113465,12684,32626,0,partial_ratio_too_low,0.0,0.239726,0.522352,1.0,1.000000,0.0,0.244535,0.68254,0.0
113466,3306,47089,0,partial_ratio_too_low,0.0,0.194444,0.431021,1.0,0.407407,0.0,0.000000,1.00000,0.0
113467,5214,6121,0,doi_and_pages_mismatch,0.0,0.250000,0.634505,1.0,0.477778,0.0,0.230114,1.00000,0.0
113468,18265,35329,1,None,0.0,1.000000,1.000000,1.0,1.000000,1.0,1.000000,1.00000,1.0


In [ ]:
# Save the exact sampled and scored pair sets so the run is reproducible later.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sample_pairs_path = RESULTS_DIR / f"sample_pairs_seed_{GLOBAL_SEED}.csv"
scored_pairs_path = RESULTS_DIR / f"scored_pairs_seed_{GLOBAL_SEED}.csv"

sample_pairs_df.to_csv(sample_pairs_path, index=False)
scored_df.to_csv(scored_pairs_path, index=False)


Saved sample pairs to c:\Coding_projects\deduplication-toolkit\notebooks\results\sample_pairs_seed_42.csv
Saved scored pairs to c:\Coding_projects\deduplication-toolkit\notebooks\results\scored_pairs_seed_42.csv


### Inspect scored feature dataframe
We use `scored_df` directly as the feature table for model training. 

In [7]:
# Train directly from raw per-field scores in scored_df
X = scored_df.drop(columns=["is_dupe", "id_a", "id_b", "early_stop_rule"], errors="ignore")
y = scored_df["is_dupe"]

In [8]:
MODEL_FACTORIES = {
    "knn": lambda: KNeighborsClassifier(),
    "svm_linear": lambda: SVC(kernel="linear"),
    "svm_default": lambda: SVC(),
    "dtree": lambda: DecisionTreeClassifier(random_state=GLOBAL_SEED),
    "naive_bayes": lambda: GaussianNB(),
    "random_forest": lambda: RandomForestClassifier(random_state=GLOBAL_SEED),
    "logistic_regression": lambda: LogisticRegression(max_iter=10000, solver="sag", random_state=GLOBAL_SEED),
}

# Change this one value when you want to swap the model used in follow-up cells.
ACTIVE_MODEL_NAME = "svm_linear"

knn = MODEL_FACTORIES["knn"]()
svm_linear = MODEL_FACTORIES["svm_linear"]()
svm_default = MODEL_FACTORIES["svm_default"]()
dtree = MODEL_FACTORIES["dtree"]()
naive_bayes = MODEL_FACTORIES["naive_bayes"]()
random_forest = MODEL_FACTORIES["random_forest"]()
logistic_regression = MODEL_FACTORIES["logistic_regression"]()
active_model = MODEL_FACTORIES[ACTIVE_MODEL_NAME]()

In [9]:
models = {"knn": knn, "svm_linear": svm_linear, "svm_default": svm_default, "dtree": dtree, "naive_bayes": naive_bayes, "random_forest": random_forest, "logistic_regression": logistic_regression}
accuracy_evaluations = ["accuracy", "precision", "recall", "f1"]

folds = 5

In [10]:
eval_set = []
conf_mats = []
for model_name, model in models.items():
    for acc_val in accuracy_evaluations:
        cross_val_prediction = cross_val_predict(model, X, y, cv=folds)
        conf_mats.append({model_name:confusion_matrix(y, cross_val_prediction)})
        acc_standard = np.mean(cross_val_score(model, X=X, y=y, cv=folds, verbose=0, scoring=acc_val))
        newline = {"model_name": model_name, "acc_val": acc_val, "standard": acc_standard}
        eval_set.append(newline)

eval_df = pd.DataFrame(eval_set)
eval_df

,model_name,acc_val,standard
0,knn,accuracy,0.999524
1,knn,precision,0.998503
2,knn,recall,0.999119
3,knn,f1,0.998811
4,svm_linear,accuracy,0.999454
5,svm_linear,precision,0.998020
6,svm_linear,recall,0.999251
7,svm_linear,f1,0.998635
8,svm_default,accuracy,0.999595
9,svm_default,precision,0.998548


### Model choice for downstream implementation: Logistic Regression

For the remainder of this notebook, we use **logistic regression** as the primary classifier.

This choice is intentional:

- it is already integrated into the current workflow (including coefficient export for `WeightSettings`);
- it provides directly interpretable feature weights for DOI/title/author/year and metadata fields;
- it is compatible with the existing scoring pipeline and downstream codebase; and
- its cross-validated performance in the comparisons above is strong enough to proceed without changing model family at this stage.

Given these practical and performance considerations, logistic regression is the baseline model


### Deriving and saving weights

We fit a logistic regression model on the full scored feature set and export its learned parameters to a JSON artifact for downstream integration.

The saved file includes:

- **`intercept`**: the model bias term;
- **`coeffs`**: a feature-to-weight mapping for all similarity fields (for example, DOI, title, authors, year, and metadata fields).

Persisting these values allows us to:

- reuse trained weights without retraining in later runs;
- version and audit model parameters across experiments;
- compare newly learned coefficients against current deduper defaults; and
- plug calibrated weights into the deduper scoring pipeline (for example, `WeightSettings`) during v2 algorithm implementation.

In [ ]:
import json

# Fit logistic regression on full dataset to extract weights
lr_full = LogisticRegression(max_iter=10000, solver="sag")
lr_full.fit(X, y)

feature_names = list(X.columns)
coeffs = dict(zip(feature_names, lr_full.coef_[0].tolist(), strict=False))
intercept = float(lr_full.intercept_[0])

results = {"intercept": intercept, "coeffs": coeffs}

# Display
coeffs_df = pd.DataFrame([
    {"feature": k, "coefficient": v}
    for k, v in sorted(coeffs.items(), key=lambda x: abs(x[1]), reverse=True)
])
print(f"Intercept: {intercept:.6f}")
display(coeffs_df)

# Save to notebooks/results/
output_dir = REPO_DIR / "notebooks" / "results"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "logistic_coeffs_new_july.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

Intercept: -18.688678


,feature,coefficient
0,title,11.591445
1,doi,5.304259
2,pages,3.791240
3,authors,3.704393
4,journal,3.179915
5,year,2.646572
6,issue,1.153251
7,abstract,-0.364041
8,volume,-0.119415


Saved to c:\Coding_projects\deduplication-toolkit\notebooks\results\logistic_coeffs_new_july.json
